# 03 — Returns and Discount Factor
**Week 3 | RL Fundamentals**

The **return** G_t is the cumulative reward from time t onwards:

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

The discount factor γ ∈ [0,1) controls how much the agent cares about future rewards.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Computing Returns

In [ ]:
def compute_returns(rewards, gamma):
    """Compute discounted returns for a list of rewards (backwards pass)."""
    G = np.zeros(len(rewards))
    running = 0.0
    for t in reversed(range(len(rewards))):
        running = rewards[t] + gamma * running
        G[t] = running
    return G

# Example trajectory: -0.1 per step, +10 at end
rewards = [-0.1] * 10 + [10.0]
t = np.arange(len(rewards))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, gamma in zip(axes, [0.5, 0.9, 0.99]):
    G = compute_returns(rewards, gamma)
    ax.bar(t, G, color='steelblue', edgecolor='white')
    ax.set_title(f'γ = {gamma}\nG_0 = {G[0]:.2f}')
    ax.set_xlabel('Time step'); ax.set_ylabel('Return G_t')
plt.suptitle('Return at each timestep for different γ', y=1.02)
plt.tight_layout(); plt.show()

## 2. Effect of γ on Discount Weights

In [ ]:
steps = np.arange(0, 50)
fig, ax = plt.subplots(figsize=(8, 3.5))
for gamma, color in [(0.5,'tomato'),(0.9,'steelblue'),(0.99,'seagreen'),(1.0,'darkorange')]:
    weights = gamma ** steps
    ax.plot(steps, weights, label=f'γ={gamma}', linewidth=2, color=color)
ax.set_xlabel('Steps into the future k')
ax.set_ylabel('Discount weight γ^k')
ax.set_title('How much do future rewards count?')
ax.legend(); plt.tight_layout(); plt.show()
print("\nA reward 20 steps away is worth:")
for g in [0.5, 0.9, 0.99, 1.0]:
    print(f"  γ={g}: {g**20:.4f} of its face value")

## 3. Episodic vs Continuing Tasks

In [ ]:
# Episodic: trajectory has a natural end (e.g. reaching the goal)
def episodic_returns(n_steps=15, goal_reward=10.0, step_cost=-0.1, gamma=0.99):
    rewards = [step_cost] * (n_steps-1) + [goal_reward]
    return compute_returns(rewards, gamma)

# Continuing: no terminal state; must use discounting to keep G finite
def continuing_returns(n_steps=50, reward_per_step=1.0, gamma=0.95):
    rewards = [reward_per_step] * n_steps
    return compute_returns(rewards, gamma)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
g_ep = episodic_returns()
axes[0].bar(range(len(g_ep)), g_ep, color='steelblue', edgecolor='white')
axes[0].set_title('Episodic Task (γ=0.99)')
axes[0].set_xlabel('Step t'); axes[0].set_ylabel('Return G_t')

g_cont = continuing_returns()
axes[1].bar(range(len(g_cont)), g_cont, color='seagreen', edgecolor='white')
axes[1].set_title('Continuing Task (γ=0.95)')
axes[1].set_xlabel('Step t'); axes[1].set_ylabel('Return G_t')
plt.tight_layout(); plt.show()

print(f"Continuing task: geometric series limit = R/(1-γ) = {1.0/(1-0.95):.1f}")
print(f"Empirical G_0 = {g_cont[0]:.2f}")

## 4. The Bellman Equation for Returns
G_t = R_{t+1} + γ * G_{t+1}  — the key recursive structure used by all RL algorithms.

In [ ]:
rewards = np.random.randn(20)  # random rewards
gamma   = 0.9
G = compute_returns(rewards, gamma)

# Verify Bellman equation: G[t] == rewards[t+1] + gamma * G[t+1]
for t in range(len(G)-1):
    lhs = G[t]
    rhs = rewards[t] + gamma * G[t+1]
    assert np.isclose(lhs, rhs), f"Mismatch at t={t}: {lhs:.4f} vs {rhs:.4f}"
print("✅ Bellman equation verified for all timesteps")

## ✅ Exercises
1. Set γ=0 and γ=1 and describe what kind of agent each creates. Is γ=1 ever safe for a continuing task?
2. Compute the optimal γ for a task where rewards are [1, 1, 1, ..., 100] after 50 steps. At what γ does the far reward contribute at least 50% of G_0?
3. **Challenge**: implement `compute_returns_forward` — compute G_t without reversing the array (hint: use the geometric series formula).

## Q1 
- γ = 0 — the agent only cares about the immediate next reward and ignores everything beyond it. It is completely myopic.
- γ = 1 — the agent treats all future rewards equally, with no discounting. This is safe for episodic tasks (finite steps), but not safe for continuing tasks — the infinite sum of rewards diverges unless rewards are zero, making G_0 infinite and mathematically undefined.

## Q2 

The rewards are `[1.0] * 50 + [100.0]` — 50 steps of reward 1, then 100 at the final step.

$$
G_0 = (\text{sum of all discounted step rewards}) + (\text{discounted final reward})
$$

$$
G_0 = \sum_{t=0}^{49} \gamma^t \cdot 1 + \gamma^{50} \cdot 100
$$

We want to find $\gamma$ such that:

$$
\frac{\gamma^{50}\times100}{G_0} \ge 0.5
$$

In [ ]:
rewards_ex2 = [1.0] * 50 + [100.0]

gammas    = np.linspace(0.5, 1.0, 10000)
fractions = np.array([(g**50 * 100) / compute_returns(rewards_ex2, g)[0] for g in gammas])

threshold_gamma = gammas[np.argmax(fractions >= 0.5)]
print(f"Threshold γ: {threshold_gamma:.4f}")

Result: The threshold is approximately γ ≈ 0.9716.

- Below this γ, the intermediate step rewards (50 × 1.0) collectively dominate G_0 — the agent effectively discounts the far reward too heavily.
- At γ ≈ 0.9716, γ⁵⁰ ≈ 0.237, meaning the far reward of 100 contributes 0.237 × 100 = 23.7 out of a G_0 of approximately 47.4 — exactly 50%.
- Above this γ, the far reward dominates and the agent becomes increasingly focused on the final large reward rather than the intermediate ones.

## Q3 

The geometric series formula computes G_t directly as a weighted dot product of future rewards

$$
G_t = \sum_{k=0}^{n-1-t} \gamma^k \cdot r_{t+k}
$$

In [ ]:
def compute_returns_forward(rewards, gamma):
    n = len(rewards)
    rewards = np.array(rewards)
    G = np.zeros(n)
    for t in range(n):
        powers = gamma ** np.arange(n - t)
        G[t]   = np.dot(powers, rewards[t:])
    return G

# Verify
rewards_test = [-0.1] * 10 + [10.0]
assert np.allclose(compute_returns(rewards_test, 0.9), compute_returns_forward(rewards_test, 0.9))
print("✅ compute_returns_forward matches compute_returns at all timesteps")